# Doctor-Patient Matching: Segmented Model with XGBoost
## Scenario A: Predicting Patient Satisfaction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb # <-- Import XGBoost
import subprocess
import sys

# Install pgeocode
try:
    import pgeocode
except ImportError:
    print("Installing pgeocode...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pgeocode"], check=True)
    import pgeocode

# --- File paths and data loading (same as before) ---
file_paths = {
    "patient": "Master_df_sample.xlsx - Patient_df.csv",
    "encounter": "Master_df_sample.xlsx - Encounter_df.csv",
    "provider": "Master_df_sample.xlsx - Provider_df.csv",
    "hospital": "Master_df_sample.xlsx - Hospital_df.csv"
}
patient_df = pd.read_csv(file_paths['patient'])
encounter_df = pd.read_csv(file_paths['encounter'])
provider_df = pd.read_csv(file_paths['provider'])
hospital_df = pd.read_csv(file_paths['hospital'])
print("Dataframes loaded successfully.")

In [ ]:
# --- Feature Engineering (same as before) ---
master_df = pd.merge(encounter_df, patient_df, on='patient_id')
master_df = pd.merge(master_df, provider_df, on='provider_id')
master_df = pd.merge(master_df, hospital_df, left_on='hospital_affiliation', right_on='hospital_id', how='left')
master_df['race_match'] = (master_df['race'] == master_df['provider_race']).astype(int)
master_df['ethnicity_match'] = (master_df['ethnicity'] == master_df['provider_ethnicity']).astype(int)
master_df['language_match'] = (master_df['language_match'] == True).astype(int)
master_df['distance_km'] = pgeocode.GeoDistance('US').query_postal_code(master_df['zip_code'].astype(str).tolist(), master_df['zip_code_hosp'].astype(str).tolist())
mean_dist_by_specialty = master_df.groupby('specialty')['distance_km'].transform('mean')
master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)
master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True)
min_distance = master_df['distance_km'].min()
max_distance = master_df['distance_km'].max()
master_df['proximity_score'] = 1 - ((master_df['distance_km'] - min_distance) / (max_distance - min_distance))
master_df['encounter_date'] = pd.to_datetime(master_df['encounter_date'])
df_sorted = master_df.sort_values(by=['patient_id', 'encounter_date'])
average_adherence = df_sorted['treatment_adherence'].mean()
df_sorted['shifted_adherence'] = df_sorted.groupby('patient_id')['treatment_adherence'].shift(1)
df_sorted['treatment_adherence_sum'] = df_sorted.groupby('patient_id')['shifted_adherence'].cumsum().fillna(0)
df_sorted['treatment_adherence_count'] = df_sorted.groupby('patient_id').cumcount()
df_sorted['historical_avg_adherence'] = np.where(df_sorted['treatment_adherence_count'] > 0, df_sorted['treatment_adherence_sum'] / df_sorted['treatment_adherence_count'], average_adherence)
master_df['historical_avg_adherence'] = df_sorted['historical_avg_adherence'].reindex_like(master_df)
print("Feature engineering complete.")

In [ ]:
# --- Training, Tuning, and Testing (XGBoost) ---
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
master_df['satisfaction_norm'] = scaler.fit_transform(master_df[['patient_satisfaction']])
target = 'satisfaction_norm'
master_df.rename(columns={'cultural_competency_rating_y': 'cultural_competency_rating_prov'}, inplace=True)
features = ['years_experience', 'cultural_competency_rating_prov', 'communication_rating', 'race_match', 'ethnicity_match', 'language_match', 'proximity_score', 'interpreter_services_24_7', 'historical_avg_adherence']
unique_preferences = master_df['cultural_preferences'].unique()
trained_models, learned_weights_dict, best_hyperparameters, test_metrics_dict = {}, {}, {}, {}

for i, preference in enumerate(unique_preferences):
    print(f"\n--- Training model for preference: {preference} ({i+1}/{len(unique_preferences)}) ---")
    segment_df = master_df[master_df['cultural_preferences'] == preference].copy()
    if len(segment_df) < 100: continue
    X_segment, y_segment = segment_df[features], segment_df[target]
    X_train_val, X_test, y_train_val, y_test = train_test_split(X_segment, y_segment, test_size=0.2, random_state=42)

    # --- Hyperparameter Tuning Section for XGBoost ---
    param_grid = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'gamma': [0, 0.1, 0.2]
    }

    xgbr = xgb.XGBRegressor(random_state=42, n_jobs=-1)
    random_search = RandomizedSearchCV(estimator=xgbr, param_distributions=param_grid, n_iter=50, cv=5, verbose=0, random_state=42, scoring='neg_mean_squared_error')

    print(f'Starting hyperparameter tuning on {len(X_train_val)} samples...')
    random_search.fit(X_train_val, y_train_val)

    print('Tuning complete. Best parameters found:')
    print(random_search.best_params_)
    best_model = random_search.best_estimator_

    # --- Final Testing Section ---
    print("\n--- Final Evaluation on the Held-Out Test Set ---")
    final_predictions = best_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
    mae = mean_absolute_error(y_test, final_predictions)
    r2 = r2_score(y_test, final_predictions)
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

    # --- Store Artifacts ---
    test_metrics = {'rmse': rmse, 'mae': mae, 'r2': r2, 'test_set_size': len(X_test)}
    learned_weights = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
    test_metrics_dict[preference], trained_models[preference], learned_weights_dict[preference], best_hyperparameters[preference] = test_metrics, best_model, learned_weights, random_search.best_params_
    print(f"--- Best model for '{preference}' trained and stored ---")